In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ワークロードの使用量

<span id="usage"></span>
使用量（Usage）は、QPU がワークロードのためにロックされている時間の量を測定したものであり、使用している実行モードによって計算方法が異なります。

* セッション使用量は、セッションがアクティブな状態のウォールクロック時間です。セッションのステータス遷移については、[セッションの長さ](/guides/run-jobs-session#session-length)を参照してください。
* バッチ使用量は、バッチ内のすべてのジョブの量子時間（QPU複合体がジョブを処理するために費やした時間）の合計です。
* 単一ジョブ使用量は、そのジョブの処理で使用される量子時間です。

なお、失敗またはキャンセルされたジョブは、特定の状況下で使用量にカウントされます。詳細は[失敗・キャンセルされたジョブ](#failed-job)のセクションをご覧ください。

有料プランのユーザーの場合、使用量によってワークロードのコストが決まります。詳細は[コストの管理](/guides/manage-cost)を参照してください。

<span id="failed-job"></span>
## 失敗・キャンセルされたジョブの使用量
ジョブが失敗またはキャンセルされた場合、報告される使用量は次のとおりです：

* ジョブまたはバッチモード：報告される使用量は、QPU がワークロードを実行するためにロックされていた時間から、失敗またはキャンセルされるまでの時間です。そのため、ロックされる前に失敗またはキャンセルが発生した場合、報告される使用量はゼロになります。それ以外の場合、ワークロードの報告使用量は、失敗またはキャンセルされるまでの使用量となります。したがって、一部の失敗したジョブは報告使用量に表示されず、他のジョブは表示される場合があります。

* セッションモード：報告される使用量は、失敗またはキャンセルされたジョブ数に関わらず、セッションがアクティブなウォールクロック時間です。

<span id="view-usage"></span>
## ワークロードの実際の使用量を照会する
ワークロードが完了した後、実際の使用量を確認するいくつかの方法があります：

- `qiskit-ibm-runtime` 0.30 以降で [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) または [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) を実行します。古いバージョンの `qiskit-ibm-runtime`（>= 0.23 かつ < 0.30）を使用している場合も、`session.details()["usage_time"]` および `batch.details()["usage_time"]` で使用量を確認できます。
- [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) を使用して、特定のバッチまたはセッションの使用量を確認します。
- [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) を使用して、単一ジョブの使用量を確認します。

<span id="instance-usage"></span>
## インスタンスの使用量を表示する
インスタンスの使用量は、[インスタンス](https://quantum.cloud.ibm.com/instances)ページ、または適切な権限を持つユーザーの場合は[アナリティクス](https://quantum.cloud.ibm.com/analytics)ページで確認できます。なお、これらのページは使用量の計算方法が異なるため、表示される使用量の数値が異なる場合があります。

インスタンスページは、現在の日の現在時刻まで、過去28日間（ローリング）のリアルタイム使用量を表示します。アナリティクスページの使用量は1時間ごとに再計算され、直近28日分の完全なデータを含みます。つまり、28日前の00:00から本日の正時までの使用量が表示されます。

## ジョブ送信前の使用量の見積もり
エラー抑制やエラー緩和のために行われる追加操作により、正確なローカル見積もりは複雑になりますが、次のベースとなる式を使って推定使用量の近似値を求めることができます：

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` は、サブジョブごとに約2秒のオーバーヘッドです。これには、制御エレクトロニクスへのペイロードの読み込みなどの操作が含まれます。プリミティブジョブは、実行エンジンが一度に処理するには大きすぎる場合、複数のサブジョブに分割されることがあります。
- `rep_delay` は[ユーザーがカスタマイズ可能](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay)なオプションであり、デフォルト値は `backend.default_rep_delay` で与えられ、ほとんどの IBM Quantum バックエンドでは250マイクロ秒です。`rep_delay` を下げると QPU の総実行時間は短縮されますが、状態準備のエラー率が増加することに注意してください。詳細は[動的繰り返しレート実行](/guides/repetition-rate-execution)ガイドをご覧ください。
- `<circuit length>` は命令の総長さです。QPU 上での各命令の実行時間は異なるため、総長はCircuitによって異なります。例えば、測定は `x` ゲートの56倍の時間を要することがあります。`backend.target[<instruction>][<qubit>].duration` を使用して、各命令の正確な継続時間を調べることができます。典型的なCircuitの長さは、50〜100マイクロ秒程度です。プリミティブでエラー抑制またはエラー緩和技術を使用している場合、Circuitに追加の命令が挿入され、回路全体の長さが増加することがあります。
    > **Note:** [実験的な `scheduler_timing` オプション](/guides/visualize-circuit-timing)は回路の総時間を返しますが、これは課金に使用される時間ではありません。
- `<num executions>` は、PUB要素がブロードキャストされた後に生成されたCircuit数にショット数を掛けた総実行数です。
    - プリミティブでエラー緩和技術を使用している場合、緩和プロセスの一部として追加のCircuitが実行され、総実行数が増加することがあります。さらに、PEAやPECなどの高度なエラー緩和技術は、ノイズ学習のためにCircuitを実行する必要があるため、はるかに高いオーバーヘッドが発生します。
    - Estimatorはクビット単位で可換な観測量をグループ化するため、実行数を削減できます。

高度なエラー緩和技術やカスタム `rep_delay` を使用していない場合は、`2+0.00035*<num executions>` を簡易計算式として使用できます。

## 次のステップ
> **Tip:** - 次のヒントを確認してください：[ジョブの実行時間を最小化する](minimize-time)。
>     - [最大実行時間](max-execution-time)を設定します。
>     - [Transpile](./transpile/)セクションでローカルTranspileの方法を学びましょう。
>     - [Transpiler設定の比較](/guides/circuit-transpilation-settings)ガイドを試してみてください。